In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

base = "/content/drive/MyDrive/Retail_Promotion_Effectiveness/Data/dunnhumby_The-Complete-Journey CSV"

print(os.listdir(base))

['campaign_desc.csv', 'campaign_table.csv', 'causal_data.csv', 'coupon.csv', 'coupon_redempt.csv', 'hh_demographic.csv', 'product.csv', 'transaction_data.csv']


In [ ]:
import pandas as pd

transaction = pd.read_csv(f"{base}/transaction_data.csv")
product = pd.read_csv(f"{base}/product.csv")

In [ ]:
transaction.shape, product.shape

((2595732, 12), (92353, 7))

In [ ]:
causal_sample = pd.read_csv(
    f"{base}/causal_data.csv",
    nrows=1000
)

causal_sample.head()

,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A
3,26190,292,70,0,A
4,26190,293,70,0,A


In [ ]:
causal_sample.shape, causal_sample.columns.tolist()

((1000, 5), ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer'])

In [ ]:
weekly_sales = (
    transaction
    .groupby(["PRODUCT_ID", "STORE_ID", "WEEK_NO"], as_index=False)
    .agg(
        sales=("SALES_VALUE", "sum"),
        units=("QUANTITY", "sum"),
        transactions=("BASKET_ID", "nunique"),
        retail_discount=("RETAIL_DISC", "sum"),
        coupon_discount=("COUPON_DISC", "sum")
    )
)

weekly_sales.shape

(2370784, 8)

In [ ]:
causal_cols = ["PRODUCT_ID", "STORE_ID", "WEEK_NO", "display", "mailer"]

causal_parts = []

for chunk in pd.read_csv(
    f"{base}/causal_data.csv",
    usecols=causal_cols,
    chunksize=500_000
):
    causal_parts.append(chunk)

causal = pd.concat(causal_parts, ignore_index=True)

print(causal.shape)

(36786524, 5)


In [ ]:
analysis_data = weekly_sales.merge(
    causal,
    on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    how="left"
)

print("Analysis rows:", len(analysis_data))
print("Rows with promotion data:", analysis_data["display"].notna().sum())
print("Rows without promotion data:", analysis_data["display"].isna().sum())

Analysis rows: 2371399
Rows with promotion data: 483935
Rows without promotion data: 1887464


In [ ]:
print("Display codes:")
print(analysis_data["display"].value_counts(dropna=False))

print("\nMailer codes:")
print(analysis_data["mailer"].value_counts(dropna=False))

Display codes:
display
NaN    1887464
0       246883
7        45301
5        36199
2        35687
3        34028
9        32601
1        17205
6        15757
A        13721
4         6553
Name: count, dtype: int64

Mailer codes:
mailer
NaN    1887464
A       195270
0       133133
D        95641
H        23665
F        23549
J         4083
L         3605
C         3421
X         1353
Z          215
Name: count, dtype: int64


In [ ]:
analysis_data["display_status"] = analysis_data["display"].apply(
    lambda x: "No Display" if x == "0"
    else "Display Present" if pd.notna(x)
    else "No Promotion Data"
)

analysis_data["mailer_status"] = analysis_data["mailer"].apply(
    lambda x: "No Mailer" if x == "0"
    else "Mailer Present" if pd.notna(x)
    else "No Promotion Data"
)

print(analysis_data["display_status"].value_counts())
print()
print(analysis_data["mailer_status"].value_counts())

display_status
No Promotion Data    1887464
No Display            246883
Display Present       237052
Name: count, dtype: int64

mailer_status
No Promotion Data    1887464
Mailer Present        350802
No Mailer             133133
Name: count, dtype: int64


In [ ]:
promotion_data = analysis_data[
    analysis_data["display"].notna()
].copy()

print("Rows with promotion data:", len(promotion_data))

print("\nAverage sales by display status:")
print(
    promotion_data.groupby("display_status")["sales"]
    .agg(["count", "mean", "median"])
)

print("\nAverage sales by mailer status:")
print(
    promotion_data.groupby("mailer_status")["sales"]
    .agg(["count", "mean", "median"])
)

Rows with promotion data: 483935

Average sales by display status:
                  count      mean  median
display_status                           
Display Present  237052  3.056100     2.0
No Display       246883  3.381255     2.2

Average sales by mailer status:
                 count      mean  median
mailer_status                           
Mailer Present  350802  3.290050     2.0
No Mailer       133133  3.042619     2.0


In [ ]:
product_store_sales = (
    promotion_data
    .groupby(["PRODUCT_ID", "STORE_ID"], as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        weeks=("WEEK_NO", "nunique")
    )
)

print(product_store_sales.head())
print("\nTotal Product-Store combinations:", len(product_store_sales))

   PRODUCT_ID  STORE_ID  total_sales  weeks
0      818980       367         2.56      3
1      818981       292         2.00      1
2      818981       304         2.00      1
3      818981       315         2.00      1
4      818981       318         2.50      1

Total Product-Store combinations: 231968


In [ ]:
print("Unique weeks:", promotion_data["WEEK_NO"].nunique())
print("Unique stores:", promotion_data["STORE_ID"].nunique())
print("Unique products:", promotion_data["PRODUCT_ID"].nunique())

print("\nWeek range:")
print(
    promotion_data["WEEK_NO"].min(),
    "to",
    promotion_data["WEEK_NO"].max()
)

Unique weeks: 93
Unique stores: 115
Unique products: 22153

Week range:
9 to 101


In [ ]:
product_store_history = (
    promotion_data
    .groupby(["PRODUCT_ID", "STORE_ID"])["WEEK_NO"]
    .nunique()
)

print(product_store_history.describe())

count    231968.000000
mean          2.083563
std           3.025383
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          74.000000
Name: WEEK_NO, dtype: float64


In [ ]:
promotion_data["promotion_type"] = "Neither"

promotion_data.loc[
    (promotion_data["display_status"] == "Display Present") &
    (promotion_data["mailer_status"] == "Mailer Present"),
    "promotion_type"
] = "Display + Mailer"

promotion_data.loc[
    (promotion_data["display_status"] == "Display Present") &
    (promotion_data["mailer_status"] == "No Mailer"),
    "promotion_type"
] = "Display Only"

promotion_data.loc[
    (promotion_data["display_status"] == "No Display") &
    (promotion_data["mailer_status"] == "Mailer Present"),
    "promotion_type"
] = "Mailer Only"

print(promotion_data["promotion_type"].value_counts())

promotion_type
Mailer Only         246883
Display Only        133133
Display + Mailer    103919
Name: count, dtype: int64


In [ ]:
promotion_type_sales = (
    promotion_data
    .groupby("promotion_type")["sales"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

print(promotion_type_sales)

                   count      mean  median
promotion_type                            
Mailer Only       246883  3.381255     2.2
Display + Mailer  103919  3.073371     2.0
Display Only      133133  3.042619     2.0


In [ ]:
# Check whether causal_data has duplicate Product + Store + Week combinations

key_cols = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]

dup_mask = causal.duplicated(key_cols, keep=False)

print("Duplicate causal rows:", dup_mask.sum())

print(
    "Duplicate Product-Store-Week keys:",
    causal.loc[dup_mask, key_cols].drop_duplicates().shape[0]
)

print("\nSample duplicate records:")

print(
    causal.loc[dup_mask]
    .sort_values(key_cols)
    .head(20)
)

Duplicate causal rows: 30490
Duplicate Product-Store-Week keys: 15245

Sample duplicate records:
       PRODUCT_ID  STORE_ID  WEEK_NO display mailer
45594       27658       361       59       0      D
45595       27658       361       59       6      D
45604       27658       361       76       0      D
45605       27658       361       76       6      D
49368       27658     34280       89       0      D
49369       27658     34280       89       6      D
49371       27658     34280       91       0      D
49372       27658     34280       91       6      D
49373       27658     34280       92       0      D
49374       27658     34280       92       6      D
49375       27658     34280       93       0      A
49376       27658     34280       93       9      A
49377       27658     34280       94       0      A
49378       27658     34280       94       9      A
49379       27658     34280       95       0      D
49380       27658     34280       95       9      D
49383       27658  

In [ ]:
# Look at how duplicate records differ in display and mailer

duplicate_data = causal.loc[
    dup_mask,
    ["PRODUCT_ID", "STORE_ID", "WEEK_NO", "display", "mailer"]
].copy()

print("Duplicate records by display + mailer combination:")

print(
    duplicate_data
    .groupby(["display", "mailer"])
    .size()
    .sort_values(ascending=False)
)

Duplicate records by display + mailer combination:
display  mailer
0        A         10681
1        A          3861
0        D          3353
7        A          3344
         D          1565
0        H           910
9        A           766
5        A           656
3        A           645
2        A           592
6        D           478
         A           422
1        D           421
7        H           284
1        H           263
3        D           260
4        A           253
2        D           249
9        D           247
0        J           137
5        H           108
A        A           107
5        D            94
0        X            68
9        H            58
7        J            51
4        D            47
6        H            46
A        H            43
3        H            42
2        0            40
0        L            39
2        H            31
4        H            30
9        J            30
6        L            29
7        X            25
3       

In [ ]:
# Check how many promotion records exist for each Product-Store-Week

duplicate_counts = (
    causal
    .groupby(["PRODUCT_ID", "STORE_ID", "WEEK_NO"])
    .size()
)

print("Records per Product-Store-Week:")
print(duplicate_counts.value_counts().sort_index())

print("\nMaximum records for one Product-Store-Week:")
print(duplicate_counts.max())

Records per Product-Store-Week:
1    36756034
2       15245
Name: count, dtype: int64

Maximum records for one Product-Store-Week:
2


In [ ]:
import pandas as pd

causal_cols = ["PRODUCT_ID", "STORE_ID", "WEEK_NO", "display", "mailer"]

causal_parts = []

for chunk in pd.read_csv(
    f"{base}/causal_data.csv",
    usecols=causal_cols,
    chunksize=500_000
):
    causal_parts.append(chunk)

causal = pd.concat(causal_parts, ignore_index=True)

print("Causal data loaded successfully!")
print("Shape:", causal.shape)

Causal data loaded successfully!
Shape: (36786524, 5)


In [ ]:
import pandas as pd

key_cols = ["PRODUCT_ID", "STORE_ID", "WEEK_NO", "display", "mailer"]

# Read causal data in smaller chunks
duplicate_parts = []

for chunk in pd.read_csv(
    f"{base}/causal_data.csv",
    usecols=key_cols,
    chunksize=500_000
):
    # Keep only duplicate Product-Store-Week records inside each chunk
    dup = chunk[chunk.duplicated(
        ["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
        keep=False
    )]

    if len(dup) > 0:
        duplicate_parts.append(dup)

causal_duplicates = pd.concat(
    duplicate_parts,
    ignore_index=True
)

print("Duplicate records found:", len(causal_duplicates))
print(
    "Duplicate Product-Store-Week keys:",
    causal_duplicates[
        ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
    ].drop_duplicates().shape[0]
)

Duplicate records found: 30490
Duplicate Product-Store-Week keys: 15245


In [ ]:
sales_keys = transaction[
    ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
].drop_duplicates()

print("Unique Product-Store-Week combinations in sales:")
print(len(sales_keys))

Unique Product-Store-Week combinations in sales:
2370784


In [ ]:
duplicate_keys = causal_duplicates[
    ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]
].drop_duplicates()

matching_keys = sales_keys.merge(
    duplicate_keys,
    on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    how="inner"
)

print("Duplicate promotion situations with actual sales:")
print(len(matching_keys))

Duplicate promotion situations with actual sales:
615


In [ ]:
print(matching_keys.head(20))

    PRODUCT_ID  STORE_ID  WEEK_NO
0      8160319       367       34
1      5569374       361       34
2       848355       343       34
3      5569230       367       34
4       974108       404       34
5      5569230       363       34
6       910291       422       34
7      1080414       310       34
8      5569230       361       34
9      5569845       361       34
10     5569230       406       34
11     1092948       367       34
12      844165       361       35
13     9527290       361       35
14     9527290       367       35
15     5569230       361       39
16      926905       361       39
17     5569471       361       39
18      917033       361       39
19     1016800       361       39


In [ ]:
promotion_615 = causal_duplicates.merge(
    matching_keys,
    on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    how="inner"
)

print("Rows:", len(promotion_615))

print(promotion_615.head(20))

Rows: 1230
    PRODUCT_ID  STORE_ID  WEEK_NO display mailer
0       826249       361       58       0      A
1       826249       361       58       9      A
2       826249       361       62       0      D
3       826249       361       62       9      D
4       826249       361       67       0      D
5       826249       361       67       6      D
6       826249       361       68       0      D
7       826249       361       68       6      D
8       826249       361       71       0      D
9       826249       361       71       9      D
10      826249       361       76       0      D
11      826249       361       76       6      D
12      826249       361       77       0      D
13      826249       361       77       6      D
14      826249       361       84       0      D
15      826249       361       84       9      D
16      826249     34280       93       0      A
17      826249     34280       93       6      A
18      826249     34280       95       0      D
19      8

In [ ]:
# Create a clean promotion summary for each Product-Store-Week

causal_summary = (
    causal_duplicates
    .groupby(["PRODUCT_ID", "STORE_ID", "WEEK_NO"], as_index=False)
    .agg(
        display_codes=("display", lambda x: "|".join(sorted(set(x.dropna().astype(str))))),
        mailer_codes=("mailer", lambda x: "|".join(sorted(set(x.dropna().astype(str)))))
    )
)

print("Clean duplicate-key summary rows:", len(causal_summary))
print(causal_summary.head(20))

Clean duplicate-key summary rows: 15245
    PRODUCT_ID  STORE_ID  WEEK_NO display_codes mailer_codes
0        27658       361       59           0|6            D
1        27658       361       76           0|6            D
2        27658     34280       89           0|6            D
3        27658     34280       91           0|6            D
4        27658     34280       92           0|6            D
5        27658     34280       93           0|9            A
6        27658     34280       94           0|9            A
7        27658     34280       95           0|9            D
8        27658     34280       98           0|9            D
9        28897       361       50           0|6            D
10       28897       361       52           0|6            D
11       28897       361       55           0|6            D
12       28897       361       56           0|6            A
13       28897       361       69           0|6            D
14       28897       361       70           0

In [ ]:
!pip -q install duckdb

import duckdb

con = duckdb.connect()

causal_summary_full = con.execute(f"""
    SELECT
        PRODUCT_ID,
        STORE_ID,
        WEEK_NO,

        MAX(CASE WHEN display != '0' THEN 1 ELSE 0 END) AS display_present,
        MAX(CASE WHEN mailer != '0' THEN 1 ELSE 0 END) AS mailer_present

    FROM read_csv_auto(
        '{base}/causal_data.csv',
        header=True
    )

    GROUP BY
        PRODUCT_ID,
        STORE_ID,
        WEEK_NO
""").df()

print("Clean causal rows:", len(causal_summary_full))
print(causal_summary_full.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clean causal rows: 36771279
   PRODUCT_ID  STORE_ID  WEEK_NO  display_present  mailer_present
0       26190       288       70                0               1
1       26190       292       70                0               1
2       26190       293       70                0               1
3       26190       295       70                0               1
4       26190       296       70                0               1


In [ ]:
weekly_sales = (
    transaction
    .groupby(
        ["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
        as_index=False
    )
    .agg(
        sales=("SALES_VALUE", "sum"),
        units=("QUANTITY", "sum"),
        transactions=("BASKET_ID", "nunique"),
        retail_discount=("RETAIL_DISC", "sum"),
        coupon_discount=("COUPON_DISC", "sum")
    )
)

print("Weekly sales rows:", len(weekly_sales))

Weekly sales rows: 2370784


In [ ]:
analysis_clean = weekly_sales.merge(
    causal_summary_full,
    on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"],
    how="left"
)

print("Rows after merge:", len(analysis_clean))
print(analysis_clean.head())

Rows after merge: 2370784
   PRODUCT_ID  STORE_ID  WEEK_NO  sales  units  transactions  retail_discount  \
0       25671      3191       59  13.96      4             1              0.0   
1       25671      3266       36   3.49      1             1              0.0   
2       25671      3313       23   3.49      1             1              0.0   
3       26081      3235       36   0.99      1             1              0.0   
4       26093     33904       66   1.59      1             1              0.0   

   coupon_discount  display_present  mailer_present  
0              0.0              NaN             NaN  
1              0.0              NaN             NaN  
2              0.0              NaN             NaN  
3              0.0              NaN             NaN  
4              0.0              NaN             NaN  


In [ ]:
print("Total weekly sales rows:", len(analysis_clean))

print("\nPromotion data availability:")
print(
    analysis_clean[["display_present", "mailer_present"]]
    .notna()
    .all(axis=1)
    .value_counts()
)

print("\nRows with promotion data:",
      analysis_clean["display_present"].notna().sum())

print("Rows without promotion data:",
      analysis_clean["display_present"].isna().sum())

Total weekly sales rows: 2370784

Promotion data availability:
False    1887464
True      483320
Name: count, dtype: int64

Rows with promotion data: 483320
Rows without promotion data: 1887464


In [ ]:
promotion_known = analysis_clean[
    analysis_clean["display_present"].notna()
].copy()

promotion_known["promotion_type"] = "Neither"

promotion_known.loc[
    (promotion_known["display_present"] == 1) &
    (promotion_known["mailer_present"] == 0),
    "promotion_type"
] = "Display Only"

promotion_known.loc[
    (promotion_known["display_present"] == 0) &
    (promotion_known["mailer_present"] == 1),
    "promotion_type"
] = "Mailer Only"

promotion_known.loc[
    (promotion_known["display_present"] == 1) &
    (promotion_known["mailer_present"] == 1),
    "promotion_type"
] = "Display + Mailer"

print(promotion_known["promotion_type"].value_counts())

promotion_type
Mailer Only         246268
Display Only        133132
Display + Mailer    103920
Name: count, dtype: int64


In [ ]:
promotion_sales_summary = (
    promotion_known
    .groupby("promotion_type")
    .agg(
        rows=("sales", "count"),
        total_sales=("sales", "sum"),
        average_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        total_units=("units", "sum"),
        average_units=("units", "mean")
    )
    .sort_values("average_sales", ascending=False)
)

promotion_sales_summary

,rows,total_sales,average_sales,median_sales,total_units,average_units
promotion_type,,,,,,
Mailer Only,246268,833296.23,3.383697,2.205,412148,1.673575
Display + Mailer,103920,319383.65,3.073361,2.000,182137,1.752666
Display Only,133132,405070.95,3.042626,2.000,200308,1.504582


In [ ]:
promotion_type_by_pair = (
    promotion_known
    .groupby(["PRODUCT_ID", "STORE_ID"])["promotion_type"]
    .nunique()
)

print("Total Product-Store pairs:", len(promotion_type_by_pair))

print(
    "Pairs with 2 or more promotion types:",
    (promotion_type_by_pair >= 2).sum()
)

print(
    "Pairs with only 1 promotion type:",
    (promotion_type_by_pair == 1).sum()
)

Total Product-Store pairs: 231968
Pairs with 2 or more promotion types: 37733
Pairs with only 1 promotion type: 194235


In [ ]:
multi_type_pairs = promotion_type_by_pair[
    promotion_type_by_pair >= 2
].index

controlled_data = promotion_known.set_index(
    ["PRODUCT_ID", "STORE_ID"]
).loc[multi_type_pairs].reset_index()

pair_type_counts = (
    controlled_data
    .groupby(["PRODUCT_ID", "STORE_ID", "promotion_type"])
    .size()
    .reset_index(name="weeks")
)

valid_pairs = (
    pair_type_counts
    .groupby(["PRODUCT_ID", "STORE_ID"])["weeks"]
    .min()
)

print("Total pairs with multiple promotion types:", len(valid_pairs))
print("Pairs where every promotion type has at least 2 weeks:", (valid_pairs >= 2).sum())

Total pairs with multiple promotion types: 37733
Pairs where every promotion type has at least 2 weeks: 7066


In [ ]:
valid_pair_list = valid_pairs[valid_pairs >= 2].index

controlled_analysis = controlled_data.set_index(
    ["PRODUCT_ID", "STORE_ID"]
).loc[valid_pair_list].reset_index()

controlled_sales = (
    controlled_analysis
    .groupby("promotion_type")
    .agg(
        weeks=("sales", "count"),
        average_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        average_units=("units", "mean")
    )
)

controlled_sales

,weeks,average_sales,median_sales,average_units
promotion_type,,,,
Display + Mailer,28627,3.443740,2.0,2.118210
Display Only,25350,3.012090,2.0,1.756963
Mailer Only,18233,3.135122,2.0,1.974716


In [ ]:
discount_summary = (
    promotion_known
    .groupby("promotion_type")
    .agg(
        average_retail_discount=("retail_discount", "mean"),
        average_coupon_discount=("coupon_discount", "mean"),
        average_sales=("sales", "mean"),
        average_units=("units", "mean")
    )
)

discount_summary

,average_retail_discount,average_coupon_discount,average_sales,average_units
promotion_type,,,,
Display + Mailer,-1.743361,-0.032891,3.073361,1.752666
Display Only,-0.651962,-0.012365,3.042626,1.504582
Mailer Only,-1.747442,-0.018296,3.383697,1.673575


In [ ]:
discount_summary = (
    promotion_known
    .groupby("promotion_type")
    .agg(
        average_retail_discount=("retail_discount", "mean"),
        average_coupon_discount=("coupon_discount", "mean"),
        average_sales=("sales", "mean"),
        average_units=("units", "mean")
    )
)

discount_summary

,average_retail_discount,average_coupon_discount,average_sales,average_units
promotion_type,,,,
Display + Mailer,-1.743361,-0.032891,3.073361,1.752666
Display Only,-0.651962,-0.012365,3.042626,1.504582
Mailer Only,-1.747442,-0.018296,3.383697,1.673575


In [ ]:
print(
    promotion_known["promotion_type"].value_counts()
)

promotion_type
Mailer Only         246268
Display Only        133132
Display + Mailer    103920
Name: count, dtype: int64


In [ ]:
category_analysis = promotion_known.merge(
    product[["PRODUCT_ID", "DEPARTMENT"]],
    on="PRODUCT_ID",
    how="left"
)

print("Rows after product merge:", len(category_analysis))
print("\nDepartments:", category_analysis["DEPARTMENT"].nunique())
print(category_analysis["DEPARTMENT"].value_counts().head(10))

Rows after product merge: 483320

Departments: 28
DEPARTMENT
GROCERY          331775
DRUG GM           37075
MEAT-PCKGD        36417
MEAT              26796
PRODUCE           26246
DELI               8482
PASTRY             6088
NUTRITION          3983
SEAFOOD-PCKGD      3606
COSMETICS           979
Name: count, dtype: int64


In [ ]:
product_promotion = (
    category_analysis
    .groupby(
        ["DEPARTMENT", "PRODUCT_ID", "promotion_type"],
        as_index=False
    )
    .agg(
        weeks=("sales", "count"),
        average_sales=("sales", "mean"),
        average_units=("units", "mean")
    )
)

product_promotion.head(20)

,DEPARTMENT,PRODUCT_ID,promotion_type,weeks,average_sales,average_units
0,CHARITABLE CONT,7167683,Display Only,1,6.740000,2.000000
1,CHEF SHOPPE,893651,Mailer Only,33,2.617273,1.151515
2,CHEF SHOPPE,1005921,Mailer Only,21,2.776190,1.238095
3,CHEF SHOPPE,1109433,Mailer Only,13,2.313077,1.076923
4,CHEF SHOPPE,16223193,Mailer Only,2,2.500000,1.000000
5,COSMETICS,819686,Mailer Only,1,3.590000,1.000000
6,COSMETICS,819967,Mailer Only,1,2.990000,1.000000
7,COSMETICS,820880,Mailer Only,1,5.420000,1.000000
8,COSMETICS,822250,Mailer Only,1,8.530000,1.000000
9,COSMETICS,822631,Mailer Only,1,2.090000,1.000000


In [ ]:
product_promotion_pivot = (
    product_promotion
    .pivot_table(
        index=["DEPARTMENT", "PRODUCT_ID"],
        columns="promotion_type",
        values=["average_sales", "weeks"],
        aggfunc="first"
    )
    .reset_index()
)

product_promotion_pivot.head(20)

DEPARTMENT PRODUCT_ID    average_sales               \
promotion_type                             Display + Mailer Display Only   
0               CHARITABLE CONT    7167683              NaN         6.74   
1                   CHEF SHOPPE     893651              NaN          NaN   
2                   CHEF SHOPPE    1005921              NaN          NaN   
3                   CHEF SHOPPE    1109433              NaN          NaN   
4                   CHEF SHOPPE   16223193              NaN          NaN   
5                     COSMETICS     819686              NaN          NaN   
6                     COSMETICS     819967              NaN          NaN   
7                     COSMETICS     820880              NaN          NaN   
8                     COSMETICS     822250              NaN          NaN   
9                     COSMETICS     822631              NaN          NaN   
10                    COSMETICS     823795              NaN          NaN   
11                    COSMETICS     825114              NaN          NaN   
12                    COSMETICS     825260              NaN          NaN   
13                    COSMETICS     825272              NaN          NaN   
14                    COSMETICS     825868              NaN          NaN   
15                    COSMETICS     826359              NaN          NaN   
16                    COSMETICS     826900              NaN         1.00   
17                    COSMETICS     828009              NaN          NaN   
18                    COSMETICS     828561              NaN          NaN   
19                    COSMETICS     830546              NaN          NaN   

                                      weeks                           
promotion_type Mailer Only Display + Mailer Display Only Mailer Only  
0                      NaN              NaN          1.0         NaN  
1                 2.617273              NaN          NaN        33.0  
2                 2.776190              NaN          NaN        21.0  
3                 2.313077              NaN          NaN        13.0  
4                 2.500000              NaN          NaN         2.0  
5                 3.590000              NaN          NaN         1.0  
6                 2.990000              NaN          NaN         1.0  
7                 5.420000              NaN          NaN         1.0  
8                 8.530000              NaN          NaN         1.0  
9                 2.090000              NaN          NaN         1.0  
10                1.450000              NaN          NaN         1.0  
11                1.790000              NaN          NaN         1.0  
12                9.290000              NaN          NaN         1.0  
13                9.290000              NaN          NaN         1.0  
14                0.990000              NaN          NaN         1.0  
15                2.406667              NaN          NaN         3.0  
16                     NaN              NaN          1.0         NaN  
17                2.652500              NaN          NaN         4.0  
18                3.590000              NaN          NaN         1.0  
19                3.590000              NaN          NaN         1.0

In [ ]:
reliable_products = product_promotion_pivot[
    product_promotion_pivot[
        [col for col in product_promotion_pivot.columns
         if col[0] == "weeks"]
    ].notna().sum(axis=1) >= 2
].copy()

print("Products with at least 2 promotion types:", len(reliable_products))
reliable_products.head(20)

Products with at least 2 promotion types: 8679


DEPARTMENT PRODUCT_ID    average_sales               \
promotion_type                            Display + Mailer Display Only   
68                   COSMETICS     885422             5.48          NaN   
73                   COSMETICS     889212              NaN         1.12   
160                  COSMETICS     989369             2.74          NaN   
291                  COSMETICS    1129094              NaN         5.49   
326                  COSMETICS    6039732              NaN         4.37   
332                  COSMETICS    6391117              NaN         5.49   
361                  COSMETICS    6552237             7.99         7.99   
364                  COSMETICS    7143616              NaN         1.83   
418                  COSMETICS    9187437             4.97         4.97   
419                  COSMETICS    9337575              NaN         5.55   
420                  COSMETICS    9396807              NaN         8.99   
422                  COSMETICS    9416370             4.97          NaN   
437                  COSMETICS    9485535              NaN         6.39   
500                  COSMETICS    9526994             2.99          NaN   
581                  COSMETICS   12352339              NaN         0.00   
599                  COSMETICS   12487417              NaN         4.79   
633                  COSMETICS   12582261              NaN        11.99   
641                  COSMETICS   13133678             4.99         9.98   
707             COUP/STR & MFG     942348             2.87          NaN   
713             COUP/STR & MFG   13877116             5.86          NaN   

                                      weeks                           
promotion_type Mailer Only Display + Mailer Display Only Mailer Only  
68                1.826667              1.0          NaN         3.0  
73                1.490000              NaN          1.0         1.0  
160               3.138000              1.0          NaN         5.0  
291               3.468000              NaN          1.0         5.0  
326               3.590000              NaN          1.0         1.0  
332               3.290000              NaN          1.0         1.0  
361                    NaN              1.0          4.0         NaN  
364               1.490000              NaN          3.0         1.0  
418               5.803333              1.0          1.0         3.0  
419               6.220000              NaN          1.0         1.0  
420               7.990000              NaN          3.0         1.0  
422               4.970000              1.0          NaN         1.0  
437               3.830000              NaN          1.0         1.0  
500               2.990000              1.0          NaN         1.0  
581               3.570000              NaN          1.0         1.0  
599               4.790000              NaN          1.0         2.0  
633               1.000000              NaN          1.0         1.0  
641                    NaN              1.0          1.0         NaN  
707               2.016190              2.0          NaN        21.0  
713               5.071667              2.0          NaN         6.0

In [ ]:
reliable_products = product_promotion_pivot[
    (
        product_promotion_pivot[
            [col for col in product_promotion_pivot.columns if col[0] == "weeks"]
        ] >= 2
    ).sum(axis=1) >= 2
].copy()

print("Reliable products:", len(reliable_products))

reliable_products.head(20)

Reliable products: 5801


DEPARTMENT PRODUCT_ID    average_sales               \
promotion_type                            Display + Mailer Display Only   
707             COUP/STR & MFG     942348         2.870000          NaN   
713             COUP/STR & MFG   13877116         5.860000          NaN   
724                       DELI     834129         1.000000     1.825556   
793                       DELI     933060         1.000000     1.990000   
801                       DELI     942556         1.250000     1.992000   
832                       DELI     995639         2.140000     2.302500   
884                       DELI    1070076         1.000000     2.140000   
898                       DELI    1093030         1.000000     1.990000   
906                       DELI    1104604         1.000000     1.990000   
908                       DELI    1106073         1.000000     1.990000   
950                       DELI    6772883              NaN     3.745000   
953                       DELI    6773180              NaN     3.500000   
962                       DELI    7024871         4.010000     4.090000   
964                       DELI    7024927         3.990000     4.090000   
965                       DELI    7025202         4.006667     4.035455   
966                       DELI    7025204              NaN     3.990000   
968                       DELI    7025496         3.990000     3.990000   
969                       DELI    7025517         3.990000     5.320000   
990                       DELI    9521679         6.990000     6.745000   
991                       DELI    9521787         5.990000     5.990000   

                                      weeks                           
promotion_type Mailer Only Display + Mailer Display Only Mailer Only  
707               2.016190              2.0          NaN        21.0  
713               5.071667              2.0          NaN         6.0  
724               1.111143              1.0          9.0        35.0  
793               1.330000              3.0          7.0        57.0  
801               1.333571              4.0          5.0        28.0  
832               2.290000              2.0          8.0         1.0  
884               1.101458              6.0         10.0        48.0  
898               1.277424              4.0          5.0        66.0  
906               1.365593              2.0          2.0        59.0  
908               1.144848              5.0          4.0        33.0  
950               3.000000              NaN          2.0         2.0  
953               3.000000              NaN          2.0         2.0  
962               4.105455              5.0          5.0        44.0  
964               4.154651              3.0          5.0        43.0  
965               4.263115              6.0         11.0        61.0  
966               4.603636              NaN          4.0        33.0  
968               4.598793              6.0          2.0        58.0  
969               4.759524              1.0          3.0        21.0  
990               8.540851              3.0          2.0        47.0  
991               7.221136              2.0          1.0        44.0

In [ ]:
comparison_summary = []

for promo_a, promo_b in [
    ("Mailer Only", "Display Only"),
    ("Mailer Only", "Display + Mailer"),
    ("Display Only", "Display + Mailer")
]:

    a = reliable_products["average_sales"][promo_a].rename("sales_a")
    b = reliable_products["average_sales"][promo_b].rename("sales_b")

    pair = pd.concat([a, b], axis=1).dropna()

    difference = pair["sales_b"] - pair["sales_a"]

    comparison_summary.append({
        "comparison": f"{promo_b} vs {promo_a}",
        "products_compared": len(pair),
        "average_difference": difference.mean(),
        "median_difference": difference.median(),
        "products_higher": (difference > 0).sum(),
        "products_lower": (difference < 0).sum()
    })

comparison_summary = pd.DataFrame(comparison_summary)

comparison_summary

,comparison,products_compared,average_difference,median_difference,products_higher,products_lower
0,Display Only vs Mailer Only,5163,0.172372,0.086929,2906,2164
1,Display + Mailer vs Mailer Only,4994,-0.003133,-0.019985,2196,2596
2,Display + Mailer vs Display Only,4506,-0.182234,-0.129346,1694,2686


In [ ]:
final_insights = comparison_summary.copy()

final_insights["absolute_difference"] = (
    final_insights["average_difference"].abs()
)

final_insights

,comparison,products_compared,average_difference,median_difference,products_higher,products_lower,absolute_difference
0,Display Only vs Mailer Only,5163,0.172372,0.086929,2906,2164,0.172372
1,Display + Mailer vs Mailer Only,4994,-0.003133,-0.019985,2196,2596,0.003133
2,Display + Mailer vs Display Only,4506,-0.182234,-0.129346,1694,2686,0.182234


In [ ]:
category_summary = (
    department_promotion
    .pivot(
        index="DEPARTMENT",
        columns="promotion_type",
        values="average_sales"
    )
    .reset_index()
)

category_summary["highest_observed_promotion"] = (
    category_summary[
        ["Display Only", "Mailer Only", "Display + Mailer"]
    ].idxmax(axis=1)
)

category_summary

promotion_type,DEPARTMENT,Display + Mailer,Display Only,Mailer Only,highest_observed_promotion
0,CHARITABLE CONT,NaN,6.740000,NaN,Display Only
1,CHEF SHOPPE,NaN,NaN,2.604928,Mailer Only
2,COSMETICS,4.408000,5.008690,3.960814,Display Only
3,COUP/STR & MFG,4.365000,1.206000,2.586389,Display + Mailer
4,DAIRY DELI,NaN,2.275455,NaN,Display Only
5,DELI,3.242969,3.279734,5.059800,Mailer Only
6,DRUG GM,2.226642,2.540312,3.525428,Mailer Only
7,FLORAL,6.851250,7.929516,9.394299,Mailer Only
8,FROZEN GROCERY,NaN,NaN,1.450000,Mailer Only
9,GARDEN CENTER,12.990000,2.000000,14.108889,Mailer Only


In [ ]:
product_choice = reliable_products.copy()

sales_cols = [
    ("average_sales", "Display Only"),
    ("average_sales", "Mailer Only"),
    ("average_sales", "Display + Mailer")
]

product_choice["highest_promotion"] = (
    product_choice[sales_cols]
    .idxmax(axis=1)
    .map({
        ("average_sales", "Display Only"): "Display Only",
        ("average_sales", "Mailer Only"): "Mailer Only",
        ("average_sales", "Display + Mailer"): "Display + Mailer"
    })
)

category_product_preference = (
    product_choice
    .groupby(["DEPARTMENT", "highest_promotion"])
    .size()
    .unstack(fill_value=0)
)

category_product_percentage = (
    category_product_preference
    .div(category_product_preference.sum(axis=1), axis=0)
    * 100
).round(2)

category_product_percentage

highest_promotion,Display + Mailer,Display Only,Mailer Only
DEPARTMENT,,,
COUP/STR & MFG,100.00,0.00,0.00
DELI,0.00,57.89,42.11
DRUG GM,19.55,45.57,34.87
FLORAL,18.18,18.18,63.64
GROCERY,25.65,41.33,33.02
MEAT,28.57,42.86,28.57
MEAT-PCKGD,27.37,42.53,30.11
NUTRITION,11.93,53.21,34.86
PASTRY,24.05,39.24,36.71


In [ ]:
# Products with all 3 promotion types available
all_three_products = product_promotion_pivot[
    (
        product_promotion_pivot[("weeks", "Display Only")] >= 2
    ) &
    (
        product_promotion_pivot[("weeks", "Mailer Only")] >= 2
    ) &
    (
        product_promotion_pivot[("weeks", "Display + Mailer")] >= 2
    )
].copy()

print("Products with all 3 promotion types:", len(all_three_products))

Products with all 3 promotion types: 3262


In [ ]:
print(all_three_products.columns)
print()
print(all_three_products.head(3))

MultiIndex([(   'DEPARTMENT',                 ''),
            (   'PRODUCT_ID',                 ''),
            ('average_sales', 'Display + Mailer'),
            ('average_sales',     'Display Only'),
            ('average_sales',      'Mailer Only'),
            (        'weeks', 'Display + Mailer'),
            (        'weeks',     'Display Only'),
            (        'weeks',      'Mailer Only')],
           names=[None, 'promotion_type'])

               DEPARTMENT PRODUCT_ID    average_sales               \
promotion_type                       Display + Mailer Display Only   
793                  DELI     933060             1.00        1.990   
801                  DELI     942556             1.25        1.992   
884                  DELI    1070076             1.00        2.140   

                                      weeks                           
promotion_type Mailer Only Display + Mailer Display Only Mailer Only  
793               1.330000              3.0          7

In [ ]:
# Create a clean comparison table for products
# where all 3 promotion types were observed

sales_comparison = all_three_products[
    [
        ("DEPARTMENT", ""),
        ("PRODUCT_ID", ""),
        ("average_sales", "Display Only"),
        ("average_sales", "Mailer Only"),
        ("average_sales", "Display + Mailer")
    ]
].copy()

sales_comparison.columns = [
    "DEPARTMENT",
    "PRODUCT_ID",
    "Display Only",
    "Mailer Only",
    "Display + Mailer"
]

# Find the highest average sales for each product
max_sales = sales_comparison[
    ["Display Only", "Mailer Only", "Display + Mailer"]
].max(axis=1)

# Count ties
sales_comparison["number_of_ties"] = (
    sales_comparison[
        ["Display Only", "Mailer Only", "Display + Mailer"]
    ]
    .eq(max_sales, axis=0)
    .sum(axis=1)
)

# Assign promotion type only when there is one clear highest value
sales_comparison["highest_promotion"] = "Tie"

sales_comparison.loc[
    (sales_comparison["Display Only"] == max_sales) &
    (sales_comparison["number_of_ties"] == 1),
    "highest_promotion"
] = "Display Only"

sales_comparison.loc[
    (sales_comparison["Mailer Only"] == max_sales) &
    (sales_comparison["number_of_ties"] == 1),
    "highest_promotion"
] = "Mailer Only"

sales_comparison.loc[
    (sales_comparison["Display + Mailer"] == max_sales) &
    (sales_comparison["number_of_ties"] == 1),
    "highest_promotion"
] = "Display + Mailer"

sales_comparison.head()

,DEPARTMENT,PRODUCT_ID,Display Only,Mailer Only,Display + Mailer,number_of_ties,highest_promotion
793,DELI,933060,1.990,1.330000,1.00,1,Display Only
801,DELI,942556,1.992,1.333571,1.25,1,Display Only
884,DELI,1070076,2.140,1.101458,1.00,1,Display Only
898,DELI,1093030,1.990,1.277424,1.00,1,Display Only
906,DELI,1104604,1.990,1.365593,1.00,1,Display Only


In [ ]:
# Count products by their highest-performing promotion type
category_product_preference = (
    sales_comparison
    .groupby(["DEPARTMENT", "highest_promotion"])
    .size()
    .unstack(fill_value=0)
)

# Convert counts into percentages within each department
category_product_percentage = (
    category_product_preference
    .div(category_product_preference.sum(axis=1), axis=0)
    * 100
).round(2)

category_product_percentage

highest_promotion,Display + Mailer,Display Only,Mailer Only,Tie
DEPARTMENT,,,,
DELI,0.00,54.55,45.45,0.00
DRUG GM,26.71,39.17,29.08,5.04
FLORAL,25.00,50.00,25.00,0.00
GROCERY,27.68,43.85,27.64,0.83
MEAT,20.00,60.00,20.00,0.00
MEAT-PCKGD,25.72,48.55,25.40,0.32
NUTRITION,9.76,46.34,43.90,0.00
PASTRY,34.48,34.48,31.03,0.00
PRODUCE,31.03,37.93,31.03,0.00


In [ ]:
department_product_counts = (
    sales_comparison
    .groupby("DEPARTMENT")["PRODUCT_ID"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="comparable_products")
)

department_product_counts

,DEPARTMENT,comparable_products
0,GROCERY,2424
1,DRUG GM,337
2,MEAT-PCKGD,311
3,SEAFOOD-PCKGD,69
4,NUTRITION,41
5,PRODUCE,29
6,PASTRY,29
7,DELI,11
8,MEAT,5
9,FLORAL,4


In [ ]:
# Keep departments with at least 30 comparable products
reliable_departments = department_product_counts[
    department_product_counts["comparable_products"] >= 30
]["DEPARTMENT"]

final_category_preference = category_product_percentage.loc[
    category_product_percentage.index.isin(reliable_departments)
].copy()

final_category_counts = department_product_counts[
    department_product_counts["DEPARTMENT"].isin(reliable_departments)
].copy()

final_category_preference

highest_promotion,Display + Mailer,Display Only,Mailer Only,Tie
DEPARTMENT,,,,
DRUG GM,26.71,39.17,29.08,5.04
GROCERY,27.68,43.85,27.64,0.83
MEAT-PCKGD,25.72,48.55,25.40,0.32
NUTRITION,9.76,46.34,43.90,0.00
SEAFOOD-PCKGD,27.54,52.17,20.29,0.00


In [ ]:
# Compare discount levels across promotion types
discount_check = (
    promotion_known
    .groupby("promotion_type")
    .agg(
        weeks=("sales", "count"),
        average_sales=("sales", "mean"),
        average_retail_discount=("retail_discount", "mean"),
        average_coupon_discount=("coupon_discount", "mean")
    )
    .round(3)
)

discount_check

,weeks,average_sales,average_retail_discount,average_coupon_discount
promotion_type,,,,
Display + Mailer,103920,3.073,-1.743,-0.033
Display Only,133132,3.043,-0.652,-0.012
Mailer Only,246268,3.384,-1.747,-0.018


In [ ]:
# Keep only the 3,262 products used in our controlled comparison

comparable_product_ids = (
    all_three_products
    .iloc[:, [0, 1]]
    .copy()
)

comparable_product_ids.columns = ["DEPARTMENT", "PRODUCT_ID"]

# Use category_analysis because it already contains DEPARTMENT
product_discount_data = category_analysis.merge(
    comparable_product_ids,
    on=["DEPARTMENT", "PRODUCT_ID"],
    how="inner"
)

# Compare sales and discount across the same comparable products
product_discount_comparison = (
    product_discount_data
    .groupby("promotion_type")
    .agg(
        comparable_weeks=("sales", "count"),
        average_sales=("sales", "mean"),
        average_retail_discount=("retail_discount", "mean"),
        average_coupon_discount=("coupon_discount", "mean")
    )
    .round(3)
)

product_discount_comparison

,comparable_weeks,average_sales,average_retail_discount,average_coupon_discount
promotion_type,,,,
Display + Mailer,93389,3.075,-1.747,-0.033
Display Only,89247,2.922,-0.709,-0.010
Mailer Only,129693,2.798,-1.453,-0.018
